# Практическая работа 9. Задание 2
Задача китайского почтальона

In [1]:
from collections import deque

# вершина: [(сосед, вес)]
graph = {
    'A': [('B', 6), ('C', 3), ('D', 8)],
    'B': [('A', 6), ('D', 5), ('E', 4)],
    'C': [('A', 3), ('D', 2), ('F', 7)],
    'D': [('A', 8), ('B', 5), ('C', 2), ('E', 3), ('F', 6)],
    'E': [('B', 4), ('D', 3), ('F', 4), ('G', 5)],
    'F': [('C', 7), ('D', 6), ('E', 4), ('G', 2)],
    'G': [('E', 5), ('F', 2)]
}
INF = float('inf')


def floyd(graph):
    # Флойд-Уоршелл, nxt нужен для восстановления пути
    vs = list(graph)
    dist = {u: {v: INF for v in vs} for u in vs}
    nxt = {u: {v: None for v in vs} for u in vs}
    for u in vs:
        dist[u][u] = 0
        for v, w in graph[u]:
            dist[u][v] = w
            nxt[u][v] = v
    for k in vs:
        for i in vs:
            for j in vs:
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
                    nxt[i][j] = nxt[i][k]
    return dist, nxt


def get_path(nxt, u, v):
    path = [u]
    while u != v:
        u = nxt[u][v]
        path.append(u)
    return path


def fleury(g, start):
    # Флери для мультиграфа: повторяющиеся ребра лежат в списке несколько раз
    g = {v: list(nb) for v, nb in g.items()}

    def is_bridge(u, v):
        g[u].remove(v)
        g[v].remove(u)
        visited = {u}
        q = deque([u])
        while q:
            x = q.popleft()
            for y in g[x]:
                if y not in visited:
                    visited.add(y)
                    q.append(y)
        g[u].append(v)
        g[v].append(u)
        return v not in visited

    route = [start]
    current = start
    while g[current]:
        next_v = None
        for nb in g[current]:
            if not is_bridge(current, nb):
                next_v = nb
                break
        if next_v is None:
            next_v = g[current][0]
        g[current].remove(next_v)
        g[next_v].remove(current)
        current = next_v
        route.append(current)
    return route

In [2]:
# 1) вершины нечетной степени
odd = [v for v in graph if len(graph[v]) % 2 != 0]
print('Вершины нечетной степени:', odd)

# 2) кратчайшие расстояния между ними
dist, nxt = floyd(graph)
print('Кратчайшие расстояния:')
for i in range(len(odd)):
    for j in range(i + 1, len(odd)):
        a, b = odd[i], odd[j]
        path = '-'.join(get_path(nxt, a, b))
        print('  d(%s,%s) = %d, путь %s' % (a, b, dist[a][b], path))

# 3) три варианта паросочетания для 4 вершин
a, b, c, d = odd
variants = [((a, b), (c, d)), ((a, c), (b, d)), ((a, d), (b, c))]
best = None
for p in variants:
    s = dist[p[0][0]][p[0][1]] + dist[p[1][0]][p[1][1]]
    print('  %s-%s и %s-%s: %d' % (p[0][0], p[0][1], p[1][0], p[1][1], s))
    if best is None or s < best[1]:
        best = (p, s)
print('Минимальное паросочетание:', best[0], ', сумма', best[1])

Вершины нечетной степени: ['A', 'B', 'C', 'D']
Кратчайшие расстояния:
  d(A,B) = 6, путь A-B
  d(A,C) = 3, путь A-C
  d(A,D) = 5, путь A-C-D
  d(B,C) = 7, путь B-D-C
  d(B,D) = 5, путь B-D
  d(C,D) = 2, путь C-D
  A-B и C-D: 8
  A-C и B-D: 8
  A-D и B-C: 12
Минимальное паросочетание: (('A', 'B'), ('C', 'D')) , сумма 8


In [3]:
# 4) дублируем ребра кратчайших путей из паросочетания
multi = {v: [nb for nb, w in graph[v]] for v in graph}
for x, y in best[0]:
    path = get_path(nxt, x, y)
    for i in range(len(path) - 1):
        multi[path[i]].append(path[i + 1])
        multi[path[i + 1]].append(path[i])
print('Степени после дублирования:', {v: len(multi[v]) for v in multi})

route = fleury(multi, 'A')
print('Эйлеров цикл:', ' -> '.join(route))

# 5) длина маршрута
weight = {(v, nb): w for v in graph for nb, w in graph[v]}
total = sum(w for v in graph for nb, w in graph[v]) // 2
print('Сумма ребер исходного графа:', total)
print('Сумма дублированных путей:', best[1])
print('Длина маршрута:', total + best[1])
length = 0
for i in range(len(route) - 1):
    length += weight[(route[i], route[i + 1])]
print('Длина, посчитанная по циклу:', length)

Степени после дублирования: {'A': 4, 'B': 4, 'C': 4, 'D': 6, 'E': 4, 'F': 4, 'G': 2}
Эйлеров цикл: A -> B -> D -> A -> C -> D -> E -> G -> F -> C -> D -> F -> E -> B -> A
Сумма ребер исходного графа: 55
Сумма дублированных путей: 8
Длина маршрута: 63
Длина, посчитанная по циклу: 63
